In [ ]:


**Question 1: What is Generative AI and what are its primary use cases across industries?**


**Answer:**
Generative AI refers to a subset of artificial intelligence models designed to generate new, original content—such as text, images, audio, or code—based on the patterns and structures they learned from vast amounts of training data.
**Primary Use Cases:**

* **Media & Entertainment:** Generating creative writing, scripts, music, and realistic visual effects.
* **Healthcare:** Accelerating drug discovery by generating novel molecular structures, and drafting synthetic medical data for research.
* **Software Development:** Auto-generating code, finding bugs, and writing software documentation.
* **Marketing & Customer Service:** Drafting personalized marketing copy, SEO content, and powering conversational chatbots.

**Question 2: Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?**


**Answer:**

* **Role of Probabilistic Modeling:** Generative models rely on probabilistic modeling to learn the underlying distribution of the training data. Instead of memorizing data, they map out a probability distribution $P(X)$, allowing them to sample from this distribution to create new data points that have high probability (i.e., look realistic).
* **Difference from Discriminative Models:**
* **Generative Models:** Learn the joint probability distribution $P(X, Y)$ (or just $P(X)$ if unlabelled). They try to understand *how* the data was generated to create new instances.
* **Discriminative Models:** Learn the conditional probability distribution $P(Y\vert{}X)$. They only care about drawing a decision boundary between classes to classify or predict an outcome (e.g., deciding if an image is a dog or a cat).



**Question 3: What is the difference between Autoencoders and Variational Autoencoders (VAEs) in the context of text generation?**


**Answer:**

* **Autoencoders (AEs):** Map input text to a fixed, deterministic vector in the latent space. Because this space is discontinuous, if you pick a random point in the latent space to decode, it will likely result in gibberish. They are good for compression, but poor for generation.
* **Variational Autoencoders (VAEs):** Map input text to a probability distribution (represented by a mean and variance) in the latent space. This forces the latent space to be continuous and smooth. By sampling a random point from this continuous distribution, the decoder can reliably generate coherent, novel text that shares properties with the training data.

**Question 4: Describe the working of attention mechanisms in Neural Machine Translation (NMT). Why are they critical?**


**Answer:**

* **Working:** In traditional Sequence-to-Sequence (Seq2Seq) models, the entire input sentence is compressed into a single, fixed-length context vector. The attention mechanism solves this bottleneck by allowing the decoder to look at the *entire* input sequence at every step of generating the output. It calculates "attention weights," which dictate how much focus (or attention) the model should give to each specific word in the source language when predicting the next word in the target language.
* **Critical Importance:** They are critical because they solve the problem of long-range dependencies. Without attention, models "forget" the beginning of long sentences. Attention allows the model to align grammar, context, and word order between completely different linguistic structures accurately.

**Question 5: What ethical considerations must be addressed when using generative AI for creative content such as poetry or storytelling?**


**Answer:**

* **Copyright and Plagiarism:** Generative models are trained on human-created works. Generating content that heavily mimics a specific author raises intellectual property concerns.
* **Bias and Stereotyping:** Models often amplify biases present in their training data, which can lead to offensive, discriminatory, or stereotypical character depictions in stories.
* **Economic Impact:** The ability to mass-produce creative content threatens the livelihood of human writers, illustrators, and poets.
* **Authenticity:** There is a philosophical debate about the value of art generated without human emotion or intent, necessitating transparency about when AI is used to author content.

---

### **Code Cell 1: VAE Text Reconstruction (Q6)**

*(Copy this into a Colab Code cell)*

```python
# Question 6: Simple VAE for Text Reconstruction[cite: 5]
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
tf.config.run_functions_eagerly(True)

# 1. Preprocess the data
dataset = [
    "The sky is blue", "The sun is bright", "The grass is green",
    "The night is dark", "The stars are shining"
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(dataset)
vocab_size = len(tokenizer.word_index) + 1
seqs = tokenizer.texts_to_sequences(dataset)
max_len = max([len(seq) for seq in seqs])
padded_seqs = pad_sequences(seqs, maxlen=max_len, padding='post')

# 2. Build Basic VAE Model
latent_dim = 2
embed_dim = 8

# Encoder
inputs = Input(shape=(max_len,))
x = Embedding(vocab_size, embed_dim, input_length=max_len)(inputs)
x = LSTM(16)(x)
z_mean = Dense(latent_dim)(x)
z_log_var = Dense(latent_dim)(x)

# Sampling function
def sampling(args):
    z_mean, z_log_var = args
    epsilon = K.random_normal(shape=(K.shape(z_mean)[0], latent_dim), mean=0., stddev=1.0)
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

z = Lambda(sampling)([z_mean, z_log_var])

# Decoder
decoder_inputs = tf.keras.layers.RepeatVector(max_len)(z)
x_decoded = LSTM(16, return_sequences=True)(decoder_inputs)
outputs = Dense(vocab_size, activation='softmax')(x_decoded)

# VAE Model compilation
vae = Model(inputs, outputs)
reconstruction_loss = tf.keras.losses.sparse_categorical_crossentropy(inputs, outputs)
reconstruction_loss *= max_len
kl_loss = 1 + z_log_var - K.square(z_mean) - K.exp(z_log_var)
kl_loss = K.sum(kl_loss, axis=-1)
kl_loss *= -0.5
vae_loss = K.mean(reconstruction_loss + kl_loss)
vae.add_loss(vae_loss)
vae.compile(optimizer='adam')

# 3. Train and Reconstruct
print("Training VAE...")
vae.fit(padded_seqs, padded_seqs, epochs=50, verbose=0) # Fast training for small dataset

# Reconstruction inference
print("\n--- Reconstruction Results ---")
predictions = vae.predict(padded_seqs)
for i in range(len(dataset)):
    pred_seq = np.argmax(predictions[i], axis=-1)
    reconstructed = " ".join([tokenizer.index_word.get(idx, "") for idx in pred_seq if idx != 0])
    print(f"Original: {dataset[i]} \nReconstructed: {reconstructed}\n")

```

---

### **Code Cell 2: Pre-trained Model Translation (Q7)**

*(Copy this into a Colab Code cell)*

```python
# Question 7: Translate English to French/German using a Pre-trained Model[cite: 5]
# Note: While GPT can translate via prompting, specialized translation models from
# huggingface (like MarianMT) are more efficient for direct translation tasks in Colab.
from transformers import pipeline

# Load translation pipelines
print("Loading translation models...")
en_to_fr = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr")
en_to_de = pipeline("translation_en_to_de", model="Helsinki-NLP/opus-mt-en-de")

# Short English paragraph
original_text = "Generative AI is transforming the way we create content. It allows machines to write, draw, and compose music autonomously."

# Translate
french_translation = en_to_fr(original_text)[0]['translation_text']
german_translation = en_to_de(original_text)[0]['translation_text']

print("\n--- Translation Results ---")
print(f"Original (English): {original_text}")
print(f"French Translation: {french_translation}")
print(f"German Translation: {german_translation}")

```

---

### **Code Cell 3: Attention-based Encoder-Decoder (Q8)**

*(Copy this into a Colab Code cell)*

```python
# Question 8: Simple Attention-based Encoder-Decoder Model (English to Spanish)[cite: 5]
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Attention, Concatenate

# Hyperparameters
vocab_size_en = 1000
vocab_size_sp = 1000
embed_dim = 64
latent_dim = 128
max_len_en = 20
max_len_sp = 20

# Encoder
encoder_inputs = Input(shape=(max_len_en,))
enc_emb = Embedding(vocab_size_en, embed_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_len_sp,))
dec_emb = Embedding(vocab_size_sp, embed_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# Attention Layer (Bahdanau-style implementation using Keras Attention)
attention_layer = Attention()
# Query is decoder output, Value is encoder output
attention_result = attention_layer([decoder_outputs, encoder_outputs])

# Concatenate attention output and decoder LSTM output
decoder_concat = Concatenate(axis=-1)([decoder_outputs, attention_result])

# Final Dense layer for prediction
decoder_dense = Dense(vocab_size_sp, activation='softmax')
decoder_outputs_final = decoder_dense(decoder_concat)

# Compile Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs_final)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Attention-Based Encoder-Decoder architecture created successfully.")
model.summary()

```

---

### **Code Cell 4: Poem Generation using GPT-2 (Q9)**

*(Copy this into a Colab Code cell)*

```python
# Question 9: Simulate poem generation with a pre-trained GPT model[cite: 5]
from transformers import pipeline, set_seed

# Initialize GPT-2 Text Generation Pipeline
generator = pipeline('text-generation', model='gpt2')
set_seed(42)

# Reference dataset provided in the question
dataset = [
    "Roses are red, violets are blue,",
    "Sugar is sweet, and so are you.",
    "The moon glows bright in silent skies,",
    "A bird sings where the soft wind sighs."
]

# Construct a prompt to simulate fine-tuning context
prompt = "\n".join(dataset) + "\nWrite a new 4 line poem in the same style:\n"

# Generate new text based on the prompt
print("Generating poem...")
output = generator(prompt, max_new_tokens=30, num_return_sequences=1, truncation=True)

print("\n--- Generated Poem ---")
# Extracting the generated part, splitting at the prompt
generated_poem = output[0]['generated_text'].replace(prompt, "")
print(generated_poem.strip())

```

---

### **Text Cell 2: System Design (Q10)**

*(Copy this text into a Colab Markdown cell)*

**Question 10: System Design - Creative Writing Assistant**


**Answer:**

If I were designing a creative writing assistant for a publishing company to generate story plots and character descriptions, here is the end-to-end approach:

**1. Model Selection:**
I would utilize a foundational Large Language Model (LLM) optimized for narrative coherence, such as a fine-tuned version of **Llama 3, GPT-4, or Claude**. These models possess vast contextual windows, which is crucial for maintaining plot continuity and tracking character arcs over long narratives.

**2. Training Data & Fine-Tuning:**

* **Base Data:** The model would leverage its pre-training on general language.
* **Domain-Specific Data:** I would fine-tune the model using the publishing company's proprietary catalog of successful books, character sheets, and plot outlines (ensuring copyright compliance).
* **Format:** The data would be structured as `Instruction/Prompt (e.g., "Describe a brooding detective") -> Output (Character profile)`.

**3. Bias Mitigation:**

* **RLHF (Reinforcement Learning from Human Feedback):** Use editors to rank generated outputs, penalizing cliches, stereotypes, or harmful biases.
* **Prompt Guardrails:** System prompts would strictly instruct the model to ensure diverse representation and avoid generating offensive or culturally insensitive content.

**4. Evaluation Methods:**

* **Automated Metrics:** While ROUGE or BLEU are okay for translation, they are poor for creativity. I would use "LLM-as-a-judge" to evaluate continuity and grammar.
* **Human-in-the-Loop (HITL):** The primary evaluation must be subjective scoring by professional editors based on creativity, coherence, and character depth.

**5. Real-World Challenges:**

* **Hallucination & Continuity:** Generative AI struggles with long-term memory. A character's eye color might change in chapter 3. *Solution:* Implement an external "Retrieval-Augmented Generation (RAG)" memory database that stores a fixed character sheet for the AI to reference on every generation.
* **Copyright Issues:** Ensuring the generated text doesn't inadvertently plagiarize copyrighted works present in the base model's training data.
* **The "Blandness" Problem:** AI tends to regress to the mean, generating predictable, formulaic plots. Getting true "creative spark" requires advanced prompting and constant human steering.